# eden-nx2 → PS5 (Colab runner)
La lógica del build vive en el repo (`scripts/`). Este notebook solo lo trae, restaura el estado de Drive y lanza el build. Ejecuta las celdas en orden; para cambiar el build, edita el repo y vuelve a ejecutar la celda 1.

In [ ]:
# 1 · Repo + Drive (restaura estado-completo.zip si existe)
REPO_URL = "https://github.com/viejunerd/eden-nx2-ps5.git"  # @param {type:"string"}
REPO_REF = "main"  # @param {type:"string"}
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
DRV = '/content/drive/MyDrive/eden-nx2-ps5'; os.makedirs(DRV, exist_ok=True)
if os.path.isdir('/content/repo/.git'):
    !git -C /content/repo fetch -q origin && git -C /content/repo checkout -q {REPO_REF} && git -C /content/repo pull -q --ff-only
else:
    !git clone -q --branch {REPO_REF} {REPO_URL} /content/repo
!git -C /content/repo log -1 --oneline
state = f'{DRV}/estado-completo.zip'; old_cache = f'{DRV}/cache-compilacion.zip'
print('En Drive:')
!ls -lh "{DRV}"
!apt-get -qq install -y unzip >/dev/null
if os.path.isdir('/content/work'):
    print('Ya hay /content/work en este entorno; no restauro nada.')
elif os.path.exists(state):
    print('Restaurando estado completo desde Drive (varios GB, unos minutos)…')
    !unzip -q -o "{state}" -d /
    !ldconfig
elif os.path.exists(old_cache):
    print('No hay estado-completo.zip; restauro solo la caché de ccache (cache-compilacion.zip).')
    !unzip -q -o "{old_cache}" -d /root
else:
    print('Sin estado previo en Drive: la primera compilación será completa (~3 h).')

In [ ]:
# 2 · Herramientas del host (se salta lo ya instalado/restaurado)
!bash /content/repo/scripts/setup-host.sh

In [ ]:
# 3 · Configuración y build en segundo plano
# head = lo que funcionó (el rebase daba conflicto en service.cpp)
MODE = "head"  # @param ["rebase", "head"]
NX2_REF = "auto"  # @param {type:"string"}
APP_TITLE = "PPSA99208"  # @param {type:"string"}
APP_NAME = "Eden NX2"  # @param {type:"string"}
PROSPERO_REF = "main"  # @param {type:"string"}
PROSPERO_REPO = ""  # @param {type:"string"}
# True = ProsperoEden y eden-nx2 salen de las ramas prospero-nx2-ps5 / eden-nx2-ps5 de REPO_URL
# (los cambios son commits ahí). False = repos de upstream + patches/ y diag-nx2.py de este repo.
USE_BRANCHES = True  # @param {type:"boolean"}
import os
env = {'WORKDIR':'/content/work','MODE':MODE,'NX2_REF':NX2_REF.strip() or 'auto','PROSPERO_REF':PROSPERO_REF,
       'SKIP_APT':'1','JOBS':str(os.cpu_count()),'APP_TITLE':APP_TITLE.strip(),'APP_NAME':APP_NAME.strip(),
       'LLVM_CONFIG':'/usr/local/bin/llvm-config','CCACHE_DIR':'/root/.cache/ccache','CCACHE_MAXSIZE':'20G'}
if PROSPERO_REPO.strip(): env['PROSPERO_REPO'] = PROSPERO_REPO.strip()
if USE_BRANCHES:
    env.update(PROSPERO_REPO=REPO_URL, PROSPERO_REF='prospero-nx2-ps5', NX2_REPO=REPO_URL,
               NX2_REF='eden-nx2-ps5', MODE='head', PATCHES='0', DIAG='0')
open('/content/build.env','w').write(''.join(f'export {k}="{v}"\n' for k,v in env.items()))
import subprocess
if subprocess.run(['pgrep','-f','build-eden-nx2-ps5.sh'],capture_output=True).returncode==0:
    print("Ya hay un build en marcha: mira el progreso en la 3b")
else:
    if os.path.exists('/content/build.exit'): os.remove('/content/build.exit')
    import glob, shutil
    # Una configuración de Mesa a medias (sin build.ninja) se borra para repetirla
    for d in glob.glob('/content/work/mihawk-vulkan-review/.deps/work/radv-*build*'):
        if os.path.isdir(d) and not os.path.exists(d + '/build.ninja'): shutil.rmtree(d, ignore_errors=True)
    p = subprocess.Popen(['bash','-c','source /content/build.env; /content/repo/scripts/build-eden-nx2-ps5.sh > /content/build.log 2>&1; echo $? > /content/build.exit'],
                     start_new_session=True, stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    open('/content/build.pid','w').write(str(p.pid))
    print("Build lanzado en segundo plano (PID %d). Ejecuta la 3b para ver el progreso." % p.pid)

In [ ]:
# 3x · PARAR el build (mata también meson/ninja/make; úsala antes de relanzar con la 3)
import os, signal, subprocess, time
try:
    pgid = int(open('/content/build.pid').read())
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try: os.killpg(pgid, sig)
        except ProcessLookupError: break
        time.sleep(3)
except FileNotFoundError:
    print('No hay /content/build.pid (el build no se lanzó con la celda 3).')
# huérfanos de un build anterior que no se lanzó con la 3 (no se matan a sí mismos)
me = {os.getpid(), os.getppid()}
for line in subprocess.run(['ps','-eo','pid,args'],capture_output=True,text=True).stdout.splitlines()[1:]:
    pid, _, args = line.strip().partition(' ')
    if int(pid) not in me and any(k in args for k in ('bin/meson','ninja','build-eden-nx2-ps5.sh')):
        print('matando', pid, args[:100]); os.kill(int(pid), signal.SIGKILL)
print('Hecho.')

In [ ]:
# 3b · Progreso
import subprocess, os, re
run = subprocess.run(['pgrep','-f','build-eden-nx2-ps5.sh'],capture_output=True).returncode==0
print('⏳ Compilando…' if run else ('exit code: '+open('/content/build.exit').read().strip() if os.path.exists('/content/build.exit') else 'sin build'))
if os.path.exists('/content/build.log'):
    log=open('/content/build.log',errors='replace').read()
    s=re.findall(r'^==> .*$',log,re.M); n=re.findall(r'\[(\d+)/(\d+)\]',log[-20000:])
    if s: print('Último paso:',s[-1])
    if n: print('ninja: %s/%s'%n[-1])
    print('\n'.join(log.splitlines()[-15:]))

In [ ]:
# 4 · Resultado: ZIP de la app a Drive + descarga, y guardar estado completo
import glob, shutil, datetime, os
base='/content/work/ProsperoEden'; want=os.environ.get('APP_TITLE') or APP_TITLE
app=f'{base}/build/release/{want}'
exit_file='/content/build.exit'
assert os.path.exists(exit_file) and open(exit_file).read().strip()=='0', 'El build no ha terminado bien en ESTE entorno (la carpeta restaurada de Drive es la compilación vieja, sin parche). Ejecuta la celda 3 y espera a que la 3b diga exit code: 0.'
assert os.path.isdir(app), 'No hay resultado; mira el log'
f=datetime.datetime.now().strftime('%Y%m%d-%H%M')
z=shutil.make_archive(f'/content/eden-nx2-ps5-{want}-{f}','zip',root_dir=os.path.dirname(app),base_dir=want)
shutil.copy(z,DRV); shutil.copy('/content/build.log',f'{DRV}/build-{f}.log')
from google.colab import files; files.download(z)
!rm -f /content/estado-completo.zip
!cd / && zip -q -r -y -1 /content/estado-completo.zip root/.cache content/work usr/local/lib/libLLVMSPIRVLib* usr/local/lib/pkgconfig/LLVMSPIRVLib.pc usr/local/include/LLVMSPIRVLib usr/local/bin/llvm-spirv
!cp /content/estado-completo.zip "{DRV}/estado-completo.zip.tmp" && mv "{DRV}/estado-completo.zip.tmp" "{DRV}/estado-completo.zip"
print('✅ ZIP y estado guardados en Drive')